# Asistente de soporte para las aplicaciones de almacén

Chatbot basado en reglas, hecho con Python y NLTK. Primera versión del asistente que
va a sustituir al formulario de soporte de las aplicaciones de almacén. 

El notebook tiene los siguientes pasos: 

1. **Sesión**: se obtienen los datos que el chat ya conoce porque se abre desde la app (usuario, aplicación, sucursal y dispositivo).
2. **Base de conocimiento**: las intenciones a detectar, sus palabras clave, su respuesta y los datos
   que hay que pedir para levantar el ticket.
3. **Preparar el texto con NLTK**: separar el mensaje en palabras y dejar cada una en su raíz.
4. **Identificar la intención**: comparar las raíces del mensaje contra las de cada intención.
5. **Flujo de la conversación**: responder, confirmar si se resolvió y si no pedir los
   datos y armar el ticket.
6. **Pruebas**: Algunos ejemplos de conversaciones distintas.
7. **Chat interactivo**: para conversar con el asistente y probarlo.



In [ ]:
import itertools
import re
import sys

import nltk
from nltk.stem import SnowballStemmer

# NLTK necesita este paquete para separar un texto en palabras.
nltk.download("punkt_tab", quiet=True)

print("Python:", sys.version.split()[0])
print("NLTK:", nltk.__version__)

## 1. Sesión

El chat se abre desde la aplicación, así que ya sabe quién es el usuario, en qué sucursal y
almacén está y desde qué dispositivo escribe. Esos datos no se le preguntan. Aquí declaramos algunos de
prueba.

In [ ]:
# Datos de prueba para la sesión de un usuario que se usa en los tests.
SESION_PRUEBA = {
    "usuario": "almacenista.prueba",
    "sucursal": "Sucursal de prueba",
    "almacen": "ALM-PRUEBA",
    "dispositivo": "SERIE-PRUEBA",
}

# Diccionario de aplicaciones que se usan en la integración con Service Desk.
APLICACIONES = {
    "cliente_recoge": "Cliente recoge",
    "inventario": "Inventario físico",
}

# Subcategoría de Service Desk que se usa cuando la intención no trae una propia.
SUBCATEGORIA_APP = {
    "cliente_recoge": ".Inventarios - Cliente recoge",
    "inventario": "Inventario Fisico - Ciclico",
}

## 2. Base de conocimiento

Primero va el mapa de `PREGUNTAS`, la lista de todo lo que el chat le puede preguntar al usuario para
completar un ticket. Después va la base de conocimiento, donde cada intención tiene estos
campos:

- **app**: en qué aplicación aplica la intención (`todas` si aplica en las dos).
- **obligatorias**: el mensaje tiene que traer al menos una de estas palabras para que la
  intención cuente.
- **extra**: palabras secundarias. Suman puntos, pero solas no alcanzan para reconocer la
  intención.
- **atencion**: `resolver` (el chat da la solución), `solicitud` (algo que el usuario pide:
  junta los datos y levanta una solicitud) o `incidente` (algo que está fallando y lo tiene que
  revisar soporte: junta los datos y levanta un incidente).
- **respuesta**: lo que el chat le contesta al usuario en cuanto reconoce la intención.
- **datos**: lo que el chat le pregunta al usuario, uno por uno, antes de armar el ticket. Son
  nombres de la lista `PREGUNTAS`. Por ejemplo, `"fecha"` hace que pregunte "¿Qué día es el
  inventario?".
- **ticket**: lo que ya se sabe del ticket sin preguntarle nada al usuario: el tipo
  (solicitud o incidente), la subcategoría, el artículo y el asunto. Son los mismos campos que
  hoy el usuario escoge en el formulario.

In [ ]:
# Preguntas que el chat le puede hacer al usuario para completar el ticket.
# Cada una tiene un nombre corto, que es el que se usa en "datos" de cada intención, y tres cosas:
#   1. cómo se llama ese dato en el ticket,
#   2. la pregunta que ve el usuario,
#   3. True si la respuesta es un número (el chat se queda solo con el número).
# Ejemplo: si una intención pide "fecha", el chat pregunta "¿Qué día es el inventario?" y lo
# que conteste el usuario va al ticket como "Fecha del inventario".
PREGUNTAS = {
    "numero_ticket": ("Ticket de cliente recoge", "¿Qué número de ticket de cliente recoge es?", True),
    "mensaje_error": ("Mensaje en pantalla", "¿Qué dice el mensaje que te sale? Si no sale ninguno, escribe 'ninguno'.", False),
    "etapa": ("Etapa donde se quedó", "¿En qué parte se quedó: surtir, verificar o entregar?", False),
    "factura": ("Factura", "¿Cuál es la factura?", False),
    "fecha": ("Fecha del inventario", "¿Qué día es el inventario?", False),
    "hora": ("Hora de inicio", "¿A qué hora empiezan?", False),
    "contacto": ("Contacto en sitio", "¿Quién será el contacto en el almacén ese día?", False),
    "numero_inventario": ("Número de inventario", "¿Qué número de inventario es?", True),
    "area": ("Área", "¿Qué área es?", False),
    "codigo": ("Código del artículo", "¿Qué código tiene el artículo? Escríbelo como viene en la etiqueta.", False),
    "ubicacion": ("Ubicación", "¿En qué ubicación está?", False),
    "version": ("Versión que pide la app", "¿Qué versión te pide la app? Si no la sabes, escribe 'no sé'.", False),
    "descripcion": ("Descripción del usuario", "Cuéntame en una frase qué está pasando.", False),
}

# Diccionario de intenciones que el chat puede reconocer
BASE_CONOCIMIENTO = {
    # ---------------- Cliente recoge ----------------
    "cr_firma_no_cierra": {
        "app": "cliente_recoge",
        "obligatorias": ["firma", "firmar", "firmo"],
        "extra": ["cerrar", "cierra", "terminar", "finalizar", "guardar", "guarda", "error"],
        "atencion": "incidente",
        "respuesta": "Si el ticket no se cierra al capturar la firma, lo tiene que revisar soporte. Levanto el reporte por ti.",
        "datos": ["numero_ticket", "mensaje_error"],
        "ticket": {"tipo": "Incidente", "subcategoria": ".Inventarios - Cliente recoge",
                   "articulo": "QR - Entrega Cliente",
                   "asunto": "No permite cerrar el ticket al capturar la firma del cliente"},
    },
    "cr_ticket_atorado": {
        "app": "cliente_recoge",
        "obligatorias": ["atorado", "atorada", "pasmado", "trabado", "avanza", "aparece", "terminar", "finalizar"],
        "extra": ["ticket", "etapa", "pendiente", "entregar", "surtir", "verificar", "escaneados"],
        "atencion": "incidente",
        "respuesta": "Un ticket de cliente recoge que no avanza o que no aparece lo tiene que revisar soporte. Levanto el reporte por ti.",
        "datos": ["numero_ticket", "etapa"],
        "ticket": {"tipo": "Incidente", "subcategoria": ".Inventarios - Cliente recoge", "articulo": None,
                   "asunto": "Ticket de cliente recoge atorado o que no aparece"},
    },
    "cr_remision": {
        "app": "cliente_recoge",
        "obligatorias": ["remision", "remisionar"],
        "extra": ["imprimir", "imprime", "genera", "generar", "error"],
        "atencion": "incidente",
        "respuesta": "Si la remisión no se genera o no se imprime, lo tiene que revisar soporte. Levanto el reporte por ti.",
        "datos": ["factura", "mensaje_error"],
        "ticket": {"tipo": "Incidente", "subcategoria": ".Inventarios - Cliente recoge", "articulo": None,
                   "asunto": "No genera o no imprime la remisión"},
    },
    # ---------------- Inventario físico ----------------
    "inv_guardia": {
        "app": "inventario",
        "obligatorias": ["guardia", "sabado", "domingo", "programado"],
        "extra": ["inventario", "apoyo", "requerir", "necesitamos"],
        "atencion": "solicitud",
        "respuesta": "Claro, levanto la solicitud de guardia de soporte para tu inventario.",
        "datos": ["fecha", "hora", "contacto"],
        "ticket": {"tipo": "Solicitud", "subcategoria": "Inventario Fisico - Ciclico",
                   "articulo": "Guardia de Soporte", "asunto": "Guardia de soporte para inventario físico"},
    },
    "inv_area_ocupada": {
        "app": "inventario",
        "obligatorias": ["ocupado", "ocupada", "bloqueado", "bloqueada"],
        "extra": ["area", "inventario", "usuario"],
        "atencion": "resolver",
        "respuesta": "El área se bloquea mientras otro usuario la tiene abierta, y se libera cuando esa persona regresa a la pantalla de Áreas pendientes. Pídele que lo haga y vuelve a intentar.",
        "datos": ["numero_inventario", "area"],
        "ticket": {"tipo": "Incidente", "subcategoria": "Inventario Fisico - Ciclico",
                   "articulo": "Falla en Proceso", "asunto": "Área de inventario ocupada por otro usuario"},
    },
    "inv_no_inicia": {
        "app": "inventario",
        "obligatorias": ["iniciar", "inicia", "entrar", "empezar", "abrir"],
        "extra": ["inventario", "conteo", "app", "aplicacion"],
        "atencion": "resolver",
        "respuesta": "La app solo te deja entrar cuando hay un inventario activo para tu almacén. Confirma con quien dio de alta el inventario que ya esté activo y vuelve a entrar.",
        "datos": ["numero_inventario", "mensaje_error"],
        "ticket": {"tipo": "Incidente", "subcategoria": "Inventario Fisico - Ciclico",
                   "articulo": "Falla en Proceso", "asunto": "No se puede iniciar el inventario"},
    },
    "inv_no_cierra": {
        "app": "inventario",
        "obligatorias": ["cerrar", "cierra", "cierre", "terminar", "finalizar", "revision"],
        "extra": ["inventario", "conteo", "area", "ubicaciones", "pendientes"],
        "atencion": "resolver",
        "respuesta": "La app no deja terminar un área mientras tenga ubicaciones pendientes. Revisa que todas las ubicaciones del área estén escaneadas y contadas, y que no tengas revisiones pendientes.",
        "datos": ["numero_inventario", "mensaje_error"],
        "ticket": {"tipo": "Incidente", "subcategoria": "Inventario Fisico - Ciclico",
                   "articulo": "Falla en Proceso", "asunto": "No se puede cerrar el inventario"},
    },
    # ---------------- Las dos aplicaciones ----------------
    "codigo_no_se_lee": {
        "app": "todas",
        "obligatorias": ["lee", "leer", "lectura", "ilegible", "escanear", "escanea"],
        "extra": ["codigo", "etiqueta", "articulo", "ubicacion"],
        "atencion": "resolver",
        # La respuesta cambia según la app desde la que escribe el usuario.
        "respuesta": {
            "cliente_recoge": "Puedes escribir el código a mano en la pantalla de escaneo en lugar de escanearlo, o escanearlo con la cámara del dispositivo.",
            "inventario": "Puedes escribir el código de barras a mano, o usar la opción de código ilegible para escoger el artículo de la lista.",
        },
        "datos": ["codigo", "ubicacion"],
        "ticket": {"tipo": "Incidente", "subcategoria": None, "articulo": None,
                   "asunto": "No se puede leer el código del artículo"},
    },
    "actualizar_version": {
        "app": "todas",
        "obligatorias": ["version", "actualizar", "actualizacion"],
        "extra": ["app", "aplicacion", "actual", "nueva", "llego"],
        "atencion": "solicitud",
        "respuesta": "Las actualizaciones se instalan solas en los dispositivos. Si a tu dispositivo no le llegó, levanto la solicitud para que te la instalen.",
        "datos": ["version"],
        "ticket": {"tipo": "Solicitud", "subcategoria": ".Dispositivos", "articulo": "Aplicaciones",
                   "asunto": "La actualización de la app no llegó al dispositivo"},
    },
}

# Saludo y despedida solo cuentan si el mensaje no trae ningún problema (o intencion).
# Así por ejemplo: "hola, no puedo cerrar el conteo" se atiende como problema y no como saludo.
CONVERSACION = {
    "saludo": {
        "obligatorias": ["hola", "buenos", "buenas", "saludos"],
        "extra": [],
        "respuesta": "Hola, soy el asistente de soporte de {app}. Cuéntame qué problema tienes y te ayudo a resolverlo, o levanto el ticket por ti.",
    },
    "despedida": {
        "obligatorias": ["gracias", "adios"],
        "extra": [],
        "respuesta": "Con gusto. Si necesitas ayuda de nuevo, solo escríbeme. ¡Hasta luego!",
    },
}

print(len(BASE_CONOCIMIENTO), "intenciones de problemas +", len(CONVERSACION), "de conversación + no entendido")

## 3. Preparar el texto con NLTK

Antes de comparar, el mensaje se limpia, pasando por los siguientes pasos:

1. Se pasa a minúsculas y se quitan los acentos.
2. NLTK lo separa en palabras.
3. Cada palabra se deja en su raíz.
   Así "conté", "contando" y "contar" quedan como `cont` y cuentan como si fuera la misma palabra.

Las palabras clave de la base de conocimiento pasan por el mismo proceso, para que las dos
partes se comparen igual.

In [ ]:
# Se deja cada palabra en su raíz para que el chat reconozca más formas de decir lo mismo.
raiz = SnowballStemmer("spanish")
SIN_ACENTOS = str.maketrans("áéíóúü", "aeiouu")

# Se quitan acentos y se pasa a minúsculas para que el chat reconozca más formas de decir lo mismo.
def quitar_acentos(texto):
    return texto.lower().translate(SIN_ACENTOS)


def preparar(texto):
    # Separa el texto en palabras con NLTK y deja cada una en su raíz.
    palabras = nltk.word_tokenize(quitar_acentos(texto), language="spanish")
    return [raiz.stem(p) for p in palabras if p.isalnum()]


# Las palabras clave también se dejan en su raíz, una sola vez al arrancar.
for info in list(BASE_CONOCIMIENTO.values()) + list(CONVERSACION.values()):
    info["raices_obligatorias"] = {raiz.stem(quitar_acentos(p)) for p in info["obligatorias"]}
    info["raices_extra"] = {raiz.stem(quitar_acentos(p)) for p in info["extra"]}

ejemplo = "Ya conté todo y no puedo cerrar el inventario"
print("Mensaje:", ejemplo)
print("Palabras:", nltk.word_tokenize(quitar_acentos(ejemplo), language="spanish"))
print("Raíces:  ", preparar(ejemplo))

## 4. Identificar la intención

Para cada intención de la app del usuario se cuentan las raíces que coinciden, con las siguientes reglas:

- Si no coincide ninguna de las **obligatorias**, la intención no cuenta.
- Cada obligatoria vale 2 puntos y cada **extra** vale 1.
- Gana la intención con más puntos. Si ninguna tiene coincidencia, se revisa si es saludo o despedida, y
  si tampoco, queda como **no entendido**.


In [ ]:
# Función para verificar si dos raíces coincide
def coincide(raiz_mensaje, raiz_clave):
    if raiz_mensaje == raiz_clave:
        return True
    # Se consideran coincidencias las palabras que tienen al menos 6 letras y que difieren en una sola letra.
    # para evitar falsos positivos, no se consideran coincidencias palabras cortas que difieren en una sola letra.
    largas = len(raiz_mensaje) >= 6 and len(raiz_clave) >= 6
    return largas and nltk.edit_distance(raiz_mensaje, raiz_clave) <= 1

# Función que devuelve las palabras clave de la intención que coinciden con el mensaje.
def coincidencias(raices_mensaje, raices_clave):
    return {clave for clave in raices_clave if any(coincide(r, clave) for r in raices_mensaje)}

# Función que calcula el puntaje de coincidencias entre el mensaje y la intención.
def puntaje(raices_mensaje, info):
    obligatorias = coincidencias(raices_mensaje, info["raices_obligatorias"])
    if not obligatorias:
        return 0
    return 2 * len(obligatorias) + len(coincidencias(raices_mensaje, info["raices_extra"]))

# Función que identifica la intención del mensaje según la app desde la que se escribe.
def identificar_intencion(mensaje, app):
    raices = preparar(mensaje)
    mejor, mejor_puntos = None, 0
    for nombre, info in BASE_CONOCIMIENTO.items():
        if info["app"] in ("todas", app):
            puntos = puntaje(raices, info)
            if puntos > mejor_puntos:
                mejor, mejor_puntos = nombre, puntos
    if mejor:
        return mejor
    for nombre, info in CONVERSACION.items():
        if puntaje(raices, info) > 0:
            return nombre
    return "no_entendido"

# Pruebas 
for mensaje, app in [
    ("no puedo cerrar el inventario, me marca áreas en revisión", "inventario"),
    ("no me deja terminar el invetario", "inventario"),
    ("el cliente ya firmó y no me deja terminar", "cliente_recoge"),
    ("no me deja terminar", "cliente_recoge"),
    ("la app se trabó y me sacó", "inventario"),
]:
    print(f"{APLICACIONES[app]:<17} | {mensaje:<58} -> {identificar_intencion(mensaje, app)}")

## 5. Flujo de la conversación y ticket

La conversación puede tener tres estados:

- **esperando_consulta**: identifica la intención del mensaje y responde.
- **confirmando_solucion**: después de dar una solución, pregunta si se resolvió. Si el usuario
  dice que no, pasa a pedir datos.
- **pidiendo_datos**: pregunta uno por uno los datos del ticket. Cuando se tiene el último se arma el ticket.

Si el chat no entiende dos veces seguidas, levanta un ticket general con la descripción del
usuario, para que no se quede sin respuesta. En cualquier momento de las preguntas, el
usuario puede escribir `cancelar`.


In [ ]:
# Conjuntos de palabras clave para identificar respuestas afirmativas y negativas.
AFIRMATIVAS = {"si", "ya", "listo", "funciono", "quedo", "pude", "resuelto", "gracias"}
NEGATIVAS = {"no", "sigue", "igual", "tampoco", "nada"}

# Contador para generar folios de tickets.
FOLIOS = itertools.count(1)

# Función que arma un ticket de Service Desk a partir de la intención y los datos que el usuario dio.
def armar_ticket(app, sesion, intencion, datos):
    if intencion == "no_entendido":
        plantilla = {"tipo": "Incidente", "subcategoria": None, "articulo": None,
                     "asunto": "Problema que el asistente no reconoció"}
    else:
        plantilla = BASE_CONOCIMIENTO[intencion]["ticket"]
    ticket = {
        "folio": f"SIM-{next(FOLIOS):03d}",
        "request_type": plantilla["tipo"],
        "subcategory": plantilla["subcategoria"] or SUBCATEGORIA_APP[app],
    }
    if plantilla["articulo"]:
        ticket["item"] = plantilla["articulo"]
    ticket["subject"] = f"{plantilla['asunto']} ({APLICACIONES[app]})"
    ticket["description"] = "; ".join(f"{PREGUNTAS[c][0]}: {v}" for c, v in datos.items())
    ticket.update(sesion)
    return ticket

# Función que devuelve un string con los datos del ticket para mostrar al usuario.
def mostrar_ticket(ticket):
    return "\n".join(f"    {campo}: {valor}" for campo, valor in ticket.items())

# Clase que mantiene el estado de la conversación con un usuario.
class Conversacion:
    def __init__(self, app, sesion=SESION_PRUEBA):
        self.app = app
        self.sesion = sesion
        self.no_entendidos = 0
        self.tickets = []
        self.intenciones = []    # lo que identificó en cada consulta, para revisar las pruebas
        self._reiniciar()

    # Reinicia la conversación a su estado inicial, para atender otra consulta.
    def _reiniciar(self):
        self.estado = "esperando_consulta"
        self.intencion = None
        self.pendientes = []
        self.datos = {}

    # Responde a un mensaje del usuario según el estado de la conversación.
    def responder(self, mensaje):
        if self.estado == "confirmando_solucion":
            return self._confirmar_solucion(mensaje)
        if self.estado == "pidiendo_datos":
            return self._recibir_dato(mensaje)
        return self._atender_consulta(mensaje)

    # Atiende un mensaje del usuario cuando se está esperando una consulta.
    def _atender_consulta(self, mensaje):
        intencion = identificar_intencion(mensaje, self.app)
        self.intenciones.append(intencion)

        # Si no se entendió, se le pregunta al usuario que lo diga con otras palabras. Si vuelve a pasar, se levanta un ticket de soporte.
        if intencion == "no_entendido":
            self.no_entendidos += 1
            if self.no_entendidos < 2:
                return ("No te entendí. ¿Me lo puedes decir con otras palabras? Por ejemplo: "
                        "'no puedo cerrar el conteo' o 'no lee el código'.")
            self.intencion = "no_entendido"
            return ("Sigo sin entenderte, así que voy a levantar un ticket para que soporte lo revise. "
                    + self._pedir_datos(["descripcion"]))

        self.no_entendidos = 0
        if intencion in CONVERSACION:
            return CONVERSACION[intencion]["respuesta"].format(app=APLICACIONES[self.app])

        info = BASE_CONOCIMIENTO[intencion]
        self.intencion = intencion
        respuesta = info["respuesta"]
        if isinstance(respuesta, dict):
            respuesta = respuesta[self.app]
        if info["atencion"] == "resolver":
            self.estado = "confirmando_solucion"
            return respuesta + " ¿Con eso se resolvió?"
        return respuesta + " " + self._pedir_datos(info["datos"])

    # Pide al usuario que confirme si su problema se resolvió o no.
    def _confirmar_solucion(self, mensaje):
        palabras = set(nltk.word_tokenize(quitar_acentos(mensaje), language="spanish"))
        if palabras & NEGATIVAS:
            datos = BASE_CONOCIMIENTO[self.intencion]["datos"]
            return "Va, entonces levanto el ticket para que soporte lo revise. " + self._pedir_datos(datos)
        if palabras & AFIRMATIVAS:
            self._reiniciar()
            return "Qué bueno. ¿Te ayudo con algo más?"
        return "¿Se resolvió? Contéstame sí o no."

    # Pide al usuario los datos que hacen falta para levantar el ticket.
    def _pedir_datos(self, datos):
        self.estado = "pidiendo_datos"
        self.pendientes = list(datos)
        self.datos = {}
        return PREGUNTAS[self.pendientes[0]][1]

    # Recibe un dato del usuario y lo guarda. Si ya se tienen todos, arma el ticket.
    def _recibir_dato(self, mensaje):
        if quitar_acentos(mensaje).strip() == "cancelar":
            self._reiniciar()
            return "Listo, cancelé el reporte. ¿Te ayudo con algo más?"

        clave = self.pendientes[0]
        _, pregunta, lleva_numero = PREGUNTAS[clave]
        valor = mensaje.strip()
        if lleva_numero:
            # De "es el inventario 1234" se queda solo con "1234".
            numero = re.search(r"\d+", mensaje)
            if not numero:
                return "Necesito el número, por ejemplo 35. " + pregunta
            valor = numero.group()

        self.datos[clave] = valor
        self.pendientes.pop(0)
        if self.pendientes:
            return PREGUNTAS[self.pendientes[0]][1]

        ticket = armar_ticket(self.app, self.sesion, self.intencion, self.datos)
        self.tickets.append(ticket)
        self._reiniciar()
        return (f"Listo, armé el ticket {ticket['folio']}. En esta versión todavía no se envía a "
                f"Service Desk. Estos son sus datos:\n{mostrar_ticket(ticket)}\n¿Te ayudo con algo más?")

## 6. Pruebas

Conversaciones de prueba. Los mensajes están basados en cómo escriben los usuarios en los tickets reales. Para cada
una se revisa que el chat haya identificado la intención esperada y que haya terminado como se esperaba: resuelto usuario o con ticket.

Después van dos **pruebas extraordinarias**, con mensajes que las reglas no estan preparadas.

In [ ]:
CONVERSACIONES_PRUEBA = [
    {"app": "inventario", "esperada": "inv_guardia", "final": "ticket", "mensajes": [
        "Hola, buenos días",
        "El sábado tenemos inventario, vamos a requerir de su apoyo",
        "sábado 10 de octubre",
        "9:00 am",
        "el jefe de almacén",
    ]},
    {"app": "inventario", "esperada": "inv_no_inicia", "final": "resuelto", "mensajes": [
        "no puedo iniciar mi inventario en el ipad",
        "sí, ya me dejó entrar",
        "gracias",
    ]},
    {"app": "inventario", "esperada": "inv_no_cierra", "final": "ticket", "mensajes": [
        "no puedo cerrar el inventario, me marca áreas en revisión",
        "no, sigue igual",
        "el inventario 1234",
        "dice que hay áreas en revisión",
    ]},
    {"app": "inventario", "esperada": "inv_area_ocupada", "final": "resuelto", "mensajes": [
        "me aparece un mensaje diciendo que el inventario está siendo ocupado por un usuario",
        "listo, ya quedó",
    ]},
    {"app": "inventario", "esperada": "codigo_no_se_lee", "final": "ticket", "mensajes": [
        "la etiqueta del artículo no se lee",
        "no, tampoco aparece en la lista",
        "ESS-0000",
        "pasillo 3",
    ]},
    {"app": "cliente_recoge", "esperada": "cr_firma_no_cierra", "final": "ticket", "mensajes": [
        "el cliente ya firmó y no me deja terminar el ticket, sale un error",
        "es el 35",
        "ORA-01722",
    ]},
    {"app": "cliente_recoge", "esperada": "cr_ticket_atorado", "final": "ticket", "mensajes": [
        "el ticket se quedó pasmado y no avanza",
        "ticket 26",
        "pendiente de entregar",
    ]},
    {"app": "cliente_recoge", "esperada": "cr_remision", "final": "ticket", "mensajes": [
        "le doy salida al material pero no genera la remisión",
        "F-00001",
        "error ending command",
    ]},
    {"app": "cliente_recoge", "esperada": "codigo_no_se_lee", "final": "resuelto", "mensajes": [
        "no puedo escanear el código del artículo",
        "sí, ya pude",
    ]},
    {"app": "cliente_recoge", "esperada": "actualizar_version", "final": "ticket", "mensajes": [
        "me dice que no tengo la versión más actual de la app",
        "no sé",
    ]},
]

PRUEBAS_LIMITE = [
    {"app": "inventario", "nota": "Pide apoyo para el inventario sin decir 'sábado' ni 'guardia'.", "mensajes": [
        "tenemos inventario la próxima semana y necesitamos apoyo",
        "el sábado tenemos inventario",
        "sábado 17 de octubre",
        "8:00 am",
        "la encargada del almacén",
    ]},
    {"app": "inventario", "nota": "Con se cerró confunde que la app se cierre con no poder cerrar el inventario.", "mensajes": [
        "la app se trabó y me sacó",
        "se cerró sola",
        "no, eso no es",
        "1234",
        "se cierra sola la app",
    ]},
    {"app": "inventario", "nota": "Dos mensajes que no reconoce: levanta un ticket general para que nadie se quede sin respuesta.", "mensajes": [
        "la pantalla se puso en blanco",
        "se queda toda blanca",
        "estaba contando el pasillo 2 y se quedó en blanco",
    ]},
]

# Función que corre una conversación de prueba y muestra los mensajes y respuestas.
def correr(conversacion_prueba, numero):
    app = conversacion_prueba["app"]
    chat = Conversacion(app)
    print(f"\n=== Conversación {numero} · {APLICACIONES[app]} ===")
    for mensaje in conversacion_prueba["mensajes"]:
        print(f"Usuario:   {mensaje}")
        print(f"Asistente: {chat.responder(mensaje)}")
    return chat

# Se corren todas las conversaciones de prueba y se guardan los resultados para revisarlos.
resultados = []
for numero, prueba in enumerate(CONVERSACIONES_PRUEBA, start=1):
    chat = correr(prueba, numero)
    # La intención de la conversación es la primera que no es saludo ni despedida.
    identificada = next((i for i in chat.intenciones if i not in CONVERSACION), "ninguna")
    final = "ticket" if chat.tickets else "resuelto"
    correcto = identificada == prueba["esperada"] and final == prueba["final"]
    resultados.append((numero, APLICACIONES[prueba["app"]], prueba["esperada"], identificada, final, correcto))

In [ ]:
# Se corren las pruebas límite para ver qué tan bien reconoce el chat los problemas.
print(f"{'#':>2}  {'App':<17} {'Esperada':<20} {'Identificada':<20} {'Terminó en':<10} OK")
for numero, app, esperada, identificada, final, correcto in resultados:
    print(f"{numero:>2}  {app:<17} {esperada:<20} {identificada:<20} {final:<10} {'sí' if correcto else 'NO'}")
aciertos = sum(r[-1] for r in resultados)
print(f"\nCorrectas: {aciertos} de {len(resultados)}")

In [ ]:
# Se corren las pruebas extraordinarias para ver qué tan bien reconoce el chat los problemas.
print("PRUEBAS DE LÍMITE")
for numero, prueba in enumerate(PRUEBAS_LIMITE, start=1):
    print(f"\n(Límite {numero}) {prueba['nota']}")
    chat = correr(prueba, f"L{numero}")
    print("Intenciones identificadas:", chat.intenciones)

## 7. Chat interactivo

Corre la celda y escribe como si fueras el almacenista. Cambia `app` a `"cliente_recoge"`
para probar la otra aplicación. Escribe `salir` para terminar.

In [ ]:
# Función que permite al usuario chatear con el asistente de soporte.
def chatear(app="inventario"):
    chat = Conversacion(app)
    print(f"Asistente de soporte · {APLICACIONES[app]} (escribe 'salir' para terminar)")
    while True:
        mensaje = input("Tú: ")
        if quitar_acentos(mensaje).strip() == "salir":
            print("Asistente: Hasta luego.")
            break
        print("Asistente:", chat.responder(mensaje))


chatear(app="inventario")